# Screen candidates, then check constraints on the original

**Optional code walkthrough for AI 101 (Module 16).** Not graded. Quiz scores do not depend on it.

**Objective.** Use predicted values to screen five design candidates. Check an objective and two constraints separately. Then re-check the chosen candidate with the original calculation, as Lesson 16.1 requires.

**Supports.** Lesson 16.1 (sections “Surrogates accelerate search, not sign-off” and “Objectives and constraints answer different questions”).

Run the cells from top to bottom. All data are synthetic and invented for teaching.


## About the data

Five invented pump design candidates were screened by a surrogate. The surrogate gives a predicted electrical power, a predicted flow rate and a predicted maximum stress for each. The objective is low power. The constraints are a flow of at least 0.015 m³/s and a stress of at most 120 MPa.

All numbers are synthetic and were invented for teaching. The surrogate values and the “original simulation” values are not from real equipment. The code does not build a surrogate. It only checks its predictions.

```
Candidate  Predicted power (W)  Predicted flow (m³/s)  Predicted stress (MPa)
A                 2400                 0.016                     110
B                 2100                 0.013                     100
C                 2250                 0.017                     125
D                 2300                 0.018                     115
E                 2600                 0.020                      90
```


## Step 1. Choose by the objective alone

The objective is the lowest predicted power. First look at what happens when you use only the objective. The code keeps the best candidate seen so far.

In [ ]:
# Step 1: lowest predicted power, ignoring the constraints
candidates = [
    ("A", 2400, 0.016, 110),
    ("B", 2100, 0.013, 100),
    ("C", 2250, 0.017, 125),
    ("D", 2300, 0.018, 115),
    ("E", 2600, 0.020,  90),
]   # (name, power in W, flow in m³/s, stress in MPa), all predicted

best = candidates[0]
for c in candidates:
    if c[1] < best[1]:
        best = c
print(f"Lowest predicted power: {best[0]} at {best[1]} W")

**Expected output**

```
Lowest predicted power: B at 2100 W
```

Candidate B has the lowest predicted power. Do not accept it yet. The objective says what you want. The constraints say what is allowed.

## Step 2. Check each constraint separately

Lesson 16.1 says that a candidate with an attractive objective can still be infeasible. Check the flow and the stress separately, with their units. A candidate is feasible only if it satisfies all constraints.

In [ ]:
# Step 2: constraint check for every candidate
min_flow = 0.015          # m³/s
max_stress = 120          # MPa

feasible = []
for name, power, flow, stress in candidates:
    reasons = []
    if flow < min_flow:
        reasons.append("flow below limit")
    if stress > max_stress:
        reasons.append("stress above limit")
    if len(reasons) == 0:
        feasible.append((name, power, flow, stress))
        print(name, "feasible")
    else:
        print(name, "not feasible:", ", ".join(reasons))

**Expected output**

```
A feasible
B not feasible: flow below limit
C not feasible: stress above limit
D feasible
E feasible
```

Candidate B, the best by the objective, violates the flow limit. Candidate C violates the stress limit. Only A, D and E are in the feasible design space.

## Step 3. Choose the best feasible candidate

Now apply the objective again, but only to the feasible candidates.

In [ ]:
# Step 3: best predicted power among the feasible candidates
best = feasible[0]
for c in feasible:
    if c[1] < best[1]:
        best = c
print(f"Best feasible candidate by predicted power: {best[0]} at {best[1]} W")

**Expected output**

```
Best feasible candidate by predicted power: D at 2300 W
```

Candidate D is the best feasible candidate according to the surrogate. This is only a prediction. The surrogate helps you screen. It does not sign off a design.

## Step 4. Re-check the chosen candidate with the original

Lesson 16.1 says to re-evaluate promising candidates with the original simulation or experiment. Here the “original” values for candidate D are invented numbers. Compare the power using the course convention r = y − ŷ, where y is the original value. Then check both constraints again with the original values.

In [ ]:
# Step 4: re-evaluate candidate D with the original calculation (synthetic values)
original_power = 2380     # W
original_flow = 0.0175    # m³/s
original_stress = 118     # MPa

predicted_power = best[1]
residual = original_power - predicted_power
print(f"Original power: {original_power} W; surrogate predicted: {predicted_power} W")
print(f"Residual (original - predicted): {residual:+.0f} W")

if original_flow >= min_flow:
    print("Flow constraint with original value: satisfied")
else:
    print("Flow constraint with original value: violated")

if original_stress <= max_stress:
    print(f"Stress constraint with original value: satisfied (margin {max_stress - original_stress} MPa)")
else:
    print(f"Stress constraint with original value: violated (over the limit by {original_stress - max_stress} MPa)")

**Expected output**

```
Original power: 2380 W; surrogate predicted: 2300 W
Residual (original - predicted): +80 W
Flow constraint with original value: satisfied
Stress constraint with original value: satisfied (margin 2 MPa)
```

The surrogate under-predicted the power by 80 W and under-predicted the stress by 3 MPa. The original values still satisfy both constraints, but the stress margin is only 2 MPa. This check is verification against the original calculation. It is not physical validation. It does not authorize a hardware claim.

## Try it

Change a value and observe. In Step 2, change `max_stress = 120` to `max_stress = 115`. Then run Step 2, Step 3 and Step 4 again, in this order.

**Expected result**

```
A feasible
B not feasible: flow below limit
C not feasible: stress above limit
D feasible
E feasible
Best feasible candidate by predicted power: D at 2300 W
Original power: 2380 W; surrogate predicted: 2300 W
Residual (original - predicted): +80 W
Flow constraint with original value: satisfied
Stress constraint with original value: violated (over the limit by 3 MPa)
```

Candidate D passes the screening, because its predicted stress is exactly 115 MPa. But the original value is 118 MPa, so D violates the tighter limit. The surrogate would have passed a candidate that the original calculation rejects. This is why promising candidates must be re-checked.

## Limits

- All numbers are synthetic. The “original” values are invented. They are not from a simulation or a real pump.
- The code does not build or train a surrogate. It checks predicted values against constraints and against given original values.
- One re-check of one candidate shows nothing about the accuracy of the surrogate elsewhere. A surrogate is supported only where its training evidence covers the inputs.
- Passing a constraint check on the original calculation is verification, not physical validation on real hardware.

Lesson 16.2 describes a mini-project with a baseline, a split and an independent physical check. This walkthrough practises only the objective, constraint and re-check habit of Lesson 16.1.